# Phase 1 — Le chiffre était vrai, la flotte est perdue

## Objectifs

Cette phase ne demande pas un entraînement de modèle. Elle demande une analyse qualitative des limites d'un simple comptage de signalements.

Le travail demandé pour le rapport comporte trois parties :

1. Expliquer ce que le chiffre du 4 juillet mesure réellement et proposer d'autres interprétations possibles.
2. Choisir trois témoignages réels qui illustrent ce qu'un comptage ne peut pas révéler.
3. Formuler la tâche de machine learning qui guidera la suite : le texte d'un témoignage entre dans le système et la forme observée en sort.


## 1. Imports

In [18]:
from pathlib import Path
import csv
import urllib.request

import pandas as pd


## 2. Configuration du projet

In [19]:
URL_DATA = (
    "https://raw.githubusercontent.com/planetsig/ufo-reports/master/"
    "csv-data/ufo-complete-geocoded-time-standardized.csv"
)

DATA_DIR = Path("../data")
OUTPUT_DIR = Path("../outputs")
PHASE1_DIR = OUTPUT_DIR / "phase_1_analyse_qualitative"

DATA_DIR.mkdir(parents=True, exist_ok=True)
PHASE1_DIR.mkdir(parents=True, exist_ok=True)

DATA_PATH = DATA_DIR / "releves_klaxo3.csv"

COLUMNS = [
    "datetime",
    "city",
    "state",
    "country",
    "shape",
    "duration_seconds",
    "duration_hours_min",
    "comments",
    "date_posted",
    "latitude",
    "longitude",
]


## 3. Téléchargement du fichier

Le fichier source est téléchargé automatiquement s'il n'est pas présent localement.

In [20]:
if not DATA_PATH.exists():
    print("Téléchargement du fichier...")
    urllib.request.urlretrieve(URL_DATA, DATA_PATH)
    print("Téléchargement terminé.")
else:
    print(f"Fichier déjà disponible : {DATA_PATH}")


Fichier déjà disponible : ..\data\releves_klaxo3.csv


## 4. Chargement robuste des relevés

Les lignes ayant exactement onze champs sont chargées dans le DataFrame principal. Les lignes mal structurées sont mises à part afin de préserver la traçabilité des données.

In [21]:
lignes_valides = []
lignes_problemes = []

with open(
    DATA_PATH,
    "r",
    encoding="utf-8",
    errors="replace",
    newline="",
) as f:
    reader = csv.reader(f)

    for numero_ligne, row in enumerate(reader, start=1):
        if len(row) == len(COLUMNS):
            lignes_valides.append(row)
        else:
            lignes_problemes.append(
                {
                    "numero_ligne": numero_ligne,
                    "nombre_champs": len(row),
                    "contenu": row,
                }
            )

df = pd.DataFrame(lignes_valides, columns=COLUMNS)

print(f"Lignes chargées : {len(df)}")
print(f"Lignes isolées : {len(lignes_problemes)}")


Lignes chargées : 88679
Lignes isolées : 196


## 5. Préparation des témoignages

Les commentaires vides sont repérés. La colonne `comments_clean` conserve un texte normalisé uniquement pour faciliter les recherches dans le notebook. Les commentaires originaux restent disponibles dans la colonne `comments`.


In [22]:
df["comments_clean"] = (
    df["comments"]
    .fillna("")
    .astype(str)
    .str.strip()
)

df["shape_clean"] = (
    df["shape"]
    .fillna("")
    .astype(str)
    .str.lower()
    .str.strip()
)

nombre_commentaires_vides = int(
    df["comments_clean"].eq("").sum()
)

print(f"Témoignages vides : {nombre_commentaires_vides}")
print(f"Témoignages non vides : {len(df) - nombre_commentaires_vides}")


Témoignages vides : 35
Témoignages non vides : 88644


## 6. Explorer des témoignages réels

Cette cellule affiche un échantillon fixe de témoignages. Lis les textes et repère des exemples illustrant des contenus qu'un simple comptage quotidien ne révèle pas : description précise, doute du témoin, mouvement, bruit, couleur, ou explication alternative.


In [23]:
pd.set_option("display.max_colwidth", None)

df.loc[
    df["comments_clean"].ne(""),
    [
        "datetime",
        "city",
        "country",
        "shape",
        "comments",
    ],
] .sample(
    n=30,
    random_state=42,
)


,datetime,city,country,shape,comments
44405,4/4/1997 01:00,regina (canada),ca,fireball,I was driving down a stretch of highway&#44 just outside of regina. I was about an hour or so from reaching moosejaw. With me was my mothe
69221,7/4/2012 22:00,puyallup,us,light,Traveling orange flame or light
6718,10/3/2005 21:30,boulder,us,chevron,Low flying&#44 silent&#44 muted white lights on a chevron shaped glider
11130,11/16/2013 23:55,jacksonville,us,unknown,Three lights were coming together and then the left and right vanished. middle light stayed a moment before vanishing too.
49310,5/26/2013 23:15,novi,us,rectangle,Yellow/orange objects seen over Novi Michigan.
2797,10/17/2003 21:00,missoula,us,circle,there was a haze around object
2530,10/16/2000 23:00,jefferson city,us,triangle,Silent triangle object&#44 very low&#44 moving north then east.
57653,6/26/2011 22:00,fayetteville,us,triangle,fayetteville nc 6-26-2011&#44 triangle ufo 10pm
3050,10/18/2005 19:30,hillsboro,us,sphere,Two orbs seen hovering in sky.
19484,1/21/2013 05:20,atlanta,us,fireball,White ball of light seen by a truck driver off of I-20 west.


## 7. Explorer quelques thèmes du contenu

Les recherches suivantes ne servent pas à entraîner un modèle. Elles servent à trouver des exemples concrets pour le rapport : témoignages mentionnant une lumière, un bruit, un mouvement ou une explication possible.


In [24]:
themes = {
    "lumiere": r"\blight|lights\b",
    "bruit": r"\bsound|noise|loud|silent\b",
    "mouvement": r"\bmoving|move|hover|flew|flying\b",
    "explication": r"\bplane|aircraft|satellite|meteor\b",
}

for nom_theme, pattern in themes.items():
    nombre = int(
        df["comments_clean"]
        .str.contains(pattern, case=False, regex=True, na=False)
        .sum()
    )
    print(f"{nom_theme} : {nombre} témoignages trouvés")


lumiere : 36329 témoignages trouvés
bruit : 7675 témoignages trouvés
mouvement : 24520 témoignages trouvés
explication : 4120 témoignages trouvés


## 8. Sélection de trois témoignages pour le rapport

Choisis trois indices affichés à l'étape précédente. Remplace les valeurs `XXX` par les index réels que tu as sélectionnés.

Les trois témoignages doivent idéalement montrer trois choses différentes, par exemple :

- une description précise de forme, couleur ou mouvement ;
- une dimension sonore ou l'absence de bruit ;
- un doute, une explication alternative ou une réaction humaine.


In [25]:
INDEX_1 = 6718
INDEX_2 = 2797
INDEX_3 = 2530

indices_selectionnes = [INDEX_1, INDEX_2, INDEX_3]

trois_temoignages = df.loc[
    indices_selectionnes,
    [
        "datetime",
        "city",
        "state",
        "country",
        "shape",
        "comments",
    ],
]

trois_temoignages


,datetime,city,state,country,shape,comments
6718,10/3/2005 21:30,boulder,co,us,chevron,Low flying&#44 silent&#44 muted white lights on a chevron shaped glider
2797,10/17/2003 21:00,missoula,mt,us,circle,there was a haze around object
2530,10/16/2000 23:00,jefferson city,mo,us,triangle,Silent triangle object&#44 very low&#44 moving north then east.


## 9. Vérification de la tâche de classification

La tâche retenue pour la suite consiste à prédire `shape` à partir de `comments`. Cette cellule donne une première vue des classes et des valeurs manquantes de la colonne cible.


In [26]:
resume_shapes = pd.DataFrame(
    {
        "nombre_releves": df["shape_clean"].value_counts(),
    }
)

print(f"Nombre de formes manquantes : {int(df['shape_clean'].eq('').sum())}")
print(f"Nombre de formes distinctes non vides : {df.loc[df['shape_clean'].ne(''), 'shape_clean'].nunique()}")

resume_shapes.head(30)


Nombre de formes manquantes : 2922
Nombre de formes distinctes non vides : 29


,nombre_releves
shape_clean,
light,17872
triangle,8489
circle,8453
fireball,6562
unknown,6319
other,6247
disk,6005
sphere,5755
oval,4119


## 10. Export des éléments sélectionnés

Les trois témoignages choisis sont exportés pour pouvoir être recopiés exactement dans `RAPPORT.md`.


In [27]:
trois_temoignages.to_csv(
    PHASE1_DIR / "trois_temoignages_selectionnes.csv",
    index=True,
)

resume_shapes.to_csv(
    PHASE1_DIR / "distribution_shapes.csv",
    index=True,
)

resume_phase1 = pd.DataFrame(
    [
        {
            "nombre_releves": len(df),
            "nombre_commentaires_vides": nombre_commentaires_vides,
            "nombre_formes_manquantes": int(df["shape_clean"].eq("").sum()),
            "nombre_formes_distinctes_non_vides": int(
                df.loc[df["shape_clean"].ne(""), "shape_clean"].nunique()
            ),
        }
    ]
)

resume_phase1.to_csv(
    PHASE1_DIR / "resume_phase1.csv",
    index=False,
)

print("Fichiers exportés dans :")
print(PHASE1_DIR)


Fichiers exportés dans :
..\outputs\phase_1_analyse_qualitative
